# SVRDD · Augmentation sweep

Amaç, standart augmentation politikasında tek bir değişikliğin yol hasarı tespitine etkisini ölçmektir. Kontrol, tamamlanmış **weight decay 0.005** koşusudur. Tüm varyantlar aynı COCO başlangıç ağırlıklarıyla **80 epoch, seed 0, batch 32 ve 640 px** çözünürlükte eğitilir. Dikey çevirme kapalıdır.

| Varyant | Kontrole göre değişiklik |
|---|---|
| `scale_025` | Rastgele ölçekleme aralığı ±%50 → ±%25 |
| `hsv_s_035` | HSV doygunluk katsayısı 0.70 → 0.35 |
| `mixup_010` | Mixup olasılığı %0 → %10 |
| `mosaic_son10` | İlk 70 epoch standart mosaic, son 10 epoch kapalı |

Her varyant ayrı ölçülür. Kayıt hedefi kişisel Drive'da `augmentation_sweep/`, ortak Drive'da `svrdd/augmentation_sweep/` klasörüdür. Her varyantın ayrı bir koşu klasörü vardır.

In [ ]:
#@title 1. Deney ortamı ve kayıt konumu
KAYIT = "Kişisel Drive" #@param ["Kişisel Drive", "Ortak Drive"]
import csv
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import uuid
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError("Colab'da Runtime → Change runtime type menüsünden GPU seç.")
gpu = torch.cuda.get_device_name(0)
print("GPU", gpu, flush=True)
drive.mount("/content/drive")

my_drive = Path("/content/drive/MyDrive")
team = my_drive / "svrdd"
if KAYIT == "Ortak Drive":
    shared_archive = team / "zenodo/SVRDD_YOLO.zip"
    control_file = team / "weight_decay/weight_decay_results.csv"
    if not (shared_archive.is_file() and control_file.is_file() and (team / "runs").is_dir()):
        raise FileNotFoundError("Ortak svrdd klasöründeki veri arşivi veya kontrol sonuçları görünmüyor.")
    sweep = team / "augmentation_sweep"
elif KAYIT == "Kişisel Drive":
    shared_archive = my_drive / "SVRDD_YOLO.zip"
    sweep = my_drive / "augmentation_sweep"
else:
    raise ValueError(f"Bilinmeyen kayıt konumu: {KAYIT}")

work = Path("/content/svrdd_augmentation")
work.mkdir(exist_ok=True)
commit = "af93934c67d861346520487c72c65f752ba94584"
repo = work / "ekip_kodu"
if not repo.is_dir():
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/wudii456/415-SVRDD-Road-Damage-Detection-Project.git", str(repo)], check=True)
subprocess.run(["git", "-C", str(repo), "checkout", "--detach", "--quiet", commit], check=True)
actual = subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()
assert actual == commit
if KAYIT == "Ortak Drive":
    control_csv = control_file.read_text(encoding="utf-8-sig")
else:
    # Ekibin tamamlanmış weight decay deneyinden yayımlanmış kontrol satırı.
    ref = "81ed1309113ae7afa785c7ba09ec997c5183aa80:analysis/weight_decay_results.csv"
    control_csv = subprocess.check_output(["git", "-C", str(repo), "show", ref], text=True)
controls = [r for r in csv.DictReader(control_csv.splitlines())
            if float(r["weight_decay"]) == 0.005 and int(r["seed"]) == 0]
if len(controls) != 1:
    raise ValueError("Kontrol tablosunda tek bir 0.005 / seed 0 satırı bekleniyor.")
control = controls[0]
if [int(control[k]) for k in ("epochs", "batch_size", "image_size", "validation_images")] != [80, 32, 640, 1000]:
    raise ValueError("Kontrol satırının eğitim veya doğrulama ayarları değişmiş.")

sweep.mkdir(exist_ok=True)
probe = sweep / f".write-check-{uuid.uuid4().hex}"
with probe.open("x") as f:
    f.write("ok")
try:
    assert probe.read_text() == "ok"
finally:
    probe.unlink()
print("Sonuç konumu:", sweep)
print("Kontrol: dört zayıf sınıf AP50 =", control["below_baseline_classes_mean_ap50"],
      "· genel mAP50:95 =", control["map50_95"], flush=True)

yolo = repo / "yolov5"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(yolo / "requirements.txt")], check=True)
os.environ["WANDB_MODE"] = "disabled"
print("Weight decay referansındaki YOLOv5 sürümü hazır:", actual[:7])

### Veri bölünmesi

SVRDD arşivi yayıncının MD5 değeriyle doğrulanır. Resmî bölünme 6.000 eğitim, 1.000 doğrulama ve 1.000 test görüntüsünden oluşur. Politika seçiminde doğrulama seti kullanılır.

In [ ]:
#@title 2. Veri ve deney yapılandırması
from pathlib import Path
import hashlib
import zipfile
import yaml


def prepare_data(work, shared_archive, names):
    work, shared_archive = Path(work).resolve(), Path(shared_archive)
    work.mkdir(parents=True, exist_ok=True)
    root = work / "SVRDD_YOLO"
    archive = work / "SVRDD_YOLO.zip"
    expected_md5 = "4b897aa7081e8d0fa07f3d330e86e936"
    ready = work / f".dataset-{expected_md5}"
    if not ready.is_file():
        if not archive.is_file():
            partial = archive.with_suffix(".part")
            if shared_archive.is_file():
                size = shared_archive.stat().st_size
                print(f"Drive → Colab: {size / 1e9:.2f} GB", flush=True)
                with shared_archive.open("rb") as src, partial.open("wb") as dst:
                    for index, chunk in enumerate(iter(lambda: src.read(8 * 1024**2), b""), 1):
                        dst.write(chunk)
                        if index % 32 == 0:
                            print(f"{dst.tell() / 1e9:.2f} / {size / 1e9:.2f} GB", flush=True)
            else:
                print("SVRDD arşivi Zenodo'dan oturum diskine indiriliyor (2,9 GB).", flush=True)
                subprocess.run(["curl", "--fail", "--location", "--retry", "3", "--connect-timeout", "30",
                                "--output", str(partial),
                                "https://zenodo.org/records/10100129/files/SVRDD_YOLO.zip?download=1"],
                               check=True)
            partial.replace(archive)
        digest = hashlib.md5()
        with archive.open("rb") as src:
            for chunk in iter(lambda: src.read(8 * 1024**2), b""):
                digest.update(chunk)
        if digest.hexdigest() != expected_md5:
            if archive.resolve() != shared_archive.resolve():
                archive.unlink()
            raise ValueError("Arşiv, Zenodo MD5 değeriyle eşleşmedi. Kaynak ZIP dosyasını kontrol et.")
        print("Arşiv doğrulandı. Görseller ve etiketler Colab diskine açılıyor.", flush=True)
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(work, members=(n for n in zf.namelist() if n.startswith("SVRDD_YOLO/")))

    config = {"path": str(work), "nc": len(names), "names": list(names)}
    seen = set()
    try:
        for split, expected in (("train", 6000), ("val", 1000), ("test", 1000)):
            lines = (root / f"{split}.txt").read_text().splitlines()
            paths = [root / line.strip().replace("\\", "/") for line in lines if line.strip()]
            if len(paths) != expected or len(set(paths)) != expected or seen.intersection(paths):
                raise ValueError(f"{split} listesinde sayı, tekrar veya bölünmeler arası çakışma hatası.")
            for path in paths:
                label = path.parent.parent / "labels" / (path.stem + ".txt")
                if not path.is_file() or not label.is_file():
                    raise FileNotFoundError(f"Görsel veya etiketi eksik: {path}")
            seen.update(paths)
            manifest = work / f"{split}.txt"
            manifest.write_text("\n".join(map(str, paths)) + "\n")
            config[split] = str(manifest)
            print(f"{split}: {len(paths)}")
        if seen != set(root.glob("*/images/*.jpg")):
            raise ValueError("Resmî listeler arşivdeki tüm görsellerle eşleşmedi.")
    except (OSError, ValueError):
        ready.unlink(missing_ok=True)
        raise
    data_yaml = work / "svrdd.yaml"
    data_yaml.write_text(yaml.safe_dump(config, sort_keys=False))
    ready.touch()
    return data_yaml


import csv
import json
import os
import subprocess
import sys
import uuid
from pathlib import Path

WEAK_CLASSES = ("longitudinal_crack", "transverse_crack", "pothole", "transverse_patch")
WEAK_MEAN = "below_baseline_classes_mean_ap50"


def save_metrics_csv(path, rows):
    temp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    try:
        with temp.open("x", newline="", encoding="utf-8") as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
        os.replace(temp, path)
    finally:
        temp.unlink(missing_ok=True)


def validation_scores(log, names, n_images):
    scores = {}
    required = set(names) | {"all"}
    # YOLOv5 prints: class, images, instances, precision, recall, AP50, AP50:95.
    for line in log.splitlines():
        fields = line.split()
        if len(fields) != 7 or fields[0] not in required or fields[1] != str(n_images):
            continue
        ap50, ap5095 = map(float, fields[-2:])
        if fields[0] in scores or not (0 <= ap50 <= 1 and 0 <= ap5095 <= 1):
            raise ValueError(f"Doğrulama tablosunda geçersiz satır: {line}")
        scores[fields[0]] = (ap50, ap5095)
    if set(scores) != required:
        raise ValueError(f"Doğrulama günlüğünde eksik sınıflar: {sorted(required - set(scores))}")
    return scores


def validate_run(run: Path, *, yolo, data_yaml, names, control, gpu):
    checkpoint = run / "weights/best.pt"
    with (run / "results.csv").open(newline="") as stream:
        reader = csv.DictReader(stream)
        reader.fieldnames = [s.strip() for s in reader.fieldnames]
        epochs = [int(row["epoch"]) for row in reader]
    if epochs != list(range(80)) or not checkpoint.is_file():
        raise RuntimeError(f"Doğrulama için 80 tamamlanmış epoch ve best.pt gerekli: {run}")
    train_gpu = json.loads((run / "run.json").read_text())["training_gpu"]
    command = [
        sys.executable, "-u", "val.py", "--weights", str(checkpoint), "--data", str(data_yaml),
        "--img", "640", "--batch-size", "32", "--task", "val",
        "--project", str(run), "--name", "validation", "--exist-ok",
    ]
    result = subprocess.run(command, cwd=yolo, text=True, errors="replace",
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    log_path = run / "validation.log"
    temp_log = run / f".validation.{uuid.uuid4().hex}.tmp"
    try:
        temp_log.write_text(result.stdout, encoding="utf-8")
        os.replace(temp_log, log_path)
    finally:
        temp_log.unlink(missing_ok=True)
    if result.returncode:
        print(result.stdout[-3000:])
        raise RuntimeError(f"Doğrulama tamamlanamadı: {log_path}")

    n_images = int(control["validation_images"])
    scores = validation_scores(result.stdout, names, n_images)
    row = dict(variant=run.name.removesuffix("_s0"), weight_decay=0.005, seed=0,
               epochs=80, batch_size=32, image_size=640, training_gpu=train_gpu,
               validation_gpu=gpu, validation_images=n_images)
    row.update({f"{name}_ap50": scores[name][0] for name in names})
    row.update({WEAK_MEAN: sum(scores[name][0] for name in WEAK_CLASSES) / len(WEAK_CLASSES),
                "map50": scores["all"][0], "map50_95": scores["all"][1],
                "worst_class_ap50": min(scores[name][0] for name in names)})
    save_metrics_csv(run / "metrics.csv", [row])
    print(f"Doğrulama sonuçları kaydedildi: {run / 'metrics.csv'}")
    return row


def display_summary(sweep, variants, control):
    control = dict(control)
    class_fields = [f for f in control if f.endswith("_ap50") and f not in {WEAK_MEAN, "worst_class_ap50"}]
    control["worst_class_ap50"] = min(float(control[f]) for f in class_fields)
    highlights = (WEAK_MEAN, "map50_95", "pothole_ap50", "map50", "worst_class_ap50")
    report = []
    waiting = []
    for key in ["default", *variants]:
        if key == "default":
            row = dict(variant=key, **{k: v for k, v in control.items() if k != "variant"})
        else:
            path = sweep / f"{key}_s0" / "metrics.csv"
            if not path.is_file():
                waiting.append(key)
                continue
            with path.open(newline="") as stream:
                records = list(csv.DictReader(stream))
            if len(records) != 1 or records[0].get("variant") != key:
                raise ValueError(f"Deney adı veya satır sayısı beklenenden farklı: {path}")
            row = records[0]
        if float(row["weight_decay"]) != 0.005 or int(row["seed"]) != 0:
            raise ValueError(f"Beklenmeyen weight decay veya seed: {key}")
        if any(not (0 <= float(row[f]) <= 1) for f in (*highlights, *class_fields)):
            raise ValueError(f"AP değeri [0,1] aralığının dışında: {key}")
        summary = {f: row[f] for f in ("variant", "weight_decay", "seed", "epochs", "batch_size",
                    "image_size", "training_gpu", "validation_gpu", "validation_images")}
        summary.update({f: float(row[f]) for f in dict.fromkeys((*highlights, *class_fields))})
        for f in highlights:
            summary[f"{f}_delta_pp"] = round(100 * (float(row[f]) - float(control[f])), 5)
        report.append(summary)

    columns = ["variant", WEAK_MEAN, f"{WEAK_MEAN}_delta_pp", "map50_95", "map50_95_delta_pp",
               "pothole_ap50", "map50", "worst_class_ap50"]
    try:
        import pandas as pd
        from IPython.display import display
    except ImportError:
        print(" | ".join(columns))
        for row in report:
            print(" | ".join(str(row[column]) for column in columns))
    else:
        display(pd.DataFrame(report)[columns].round(5))
    if waiting:
        print("Doğrulaması beklenen deneyler:", ", ".join(waiting))
        print("Nihai özet tüm doğrulamalar tamamlandığında oluşturulur.")
    else:
        output = sweep / "augmentation_sweep_results.csv"
        save_metrics_csv(output, report)
        print(f"Tarama tamamlandı. Tüm sınıflar, ham ölçümler ve yüzde puan farkları: {output}")
    return report


(yolo / "train_close_mosaic.py").write_text(r'''"""Run the SVRDD 80-epoch seed-0 trial with mosaic off for the last 10 epochs."""

import os
from pathlib import Path
from unittest.mock import patch

import train
from utils import dataloaders
from utils.torch_utils import smart_resume


EPOCHS = 80
CLOSE_LAST = 10
START_EPOCH = int(os.environ.get("SVRDD_START_EPOCH", "0"))


class MosaicScheduleSampler(dataloaders._RepeatSampler):
    def __iter__(self):
        # Tag indices before worker prefetch, including batches spanning the cutoff.
        for _ in range(max(0, EPOCHS - CLOSE_LAST - START_EPOCH)):
            yield from self.sampler
        while True:
            for batch in self.sampler:
                yield [(index, False) for index in batch]


class MosaicScheduleDataset(dataloaders.LoadImagesAndLabels):
    def __getitem__(self, index):
        if not isinstance(index, tuple):
            return super().__getitem__(index)
        index, mosaic = index
        previous = self.mosaic
        self.mosaic = mosaic
        try:
            return super().__getitem__(index)
        finally:
            self.mosaic = previous


def create_loader(*args, **kwargs):
    if not kwargs.get("augment", False):
        return dataloaders.create_dataloader(*args, **kwargs)
    if kwargs["hyp"]["mosaic"] != 1.0 or kwargs["hyp"]["mixup"] != 0:
        raise ValueError("The close-mosaic trial requires initial mosaic=1.0 and mixup=0.")
    with patch.object(dataloaders, "_RepeatSampler", MosaicScheduleSampler), patch.object(
        dataloaders, "LoadImagesAndLabels", MosaicScheduleDataset
    ):
        loader, dataset = dataloaders.create_dataloader(*args, **kwargs)
    if not isinstance(loader.batch_sampler, MosaicScheduleSampler) or not dataset.mosaic:
        raise RuntimeError("Training loader did not activate the mosaic schedule.")
    print("Training mosaic on for epochs 1–70 and off for 71–80.", flush=True)
    return loader, dataset


def checked_resume(*args, **kwargs):
    state = smart_resume(*args, **kwargs)
    if state[1:] != (START_EPOCH, EPOCHS):
        raise ValueError("Checkpoint epoch or training budget differs from the verified run.")
    return state


if __name__ == "__main__":
    options = train.parse_opt()
    if options.epochs != EPOCHS or options.seed != 0 or options.rect or options.image_weights:
        raise ValueError("Use an 80-epoch seed-0 run with rectangular training and image weighting disabled.")
    if options.resume:
        if not 1 <= START_EPOCH < EPOCHS or not isinstance(options.resume, str):
            raise ValueError("Resume requires the previous last.pt and 1–79 completed epochs.")
        last = Path(options.resume)
        if last.name != "last.pt" or not last.is_file():
            raise ValueError("The specified last.pt must exist.")
    elif START_EPOCH != 0:
        raise ValueError("A fresh run must start at epoch zero.")
    if int(os.environ.get("WORLD_SIZE", "1")) != 1:
        raise ValueError("The SVRDD sweep uses one GPU.")
    with patch.object(train, "create_dataloader", create_loader), patch.object(train, "smart_resume", checked_resume):
        train.main(options)
''', encoding="utf-8")

names = ["longitudinal_crack", "transverse_crack", "alligator_crack", "pothole",
         "manhole_cover", "longitudinal_patch", "transverse_patch"]
variants = {
    "scale_025": {"scale": 0.25},
    "hsv_s_035": {"hsv_s": 0.35},
    "mixup_010": {"mixup": 0.10},
    "mosaic_son10": {},
}
data_yaml = prepare_data(work, shared_archive, names)
control_hyp = yaml.safe_load((yolo / "data/hyps/hyp.scratch-low.yaml").read_text())
expected = {"scale": 0.5, "hsv_h": 0.015, "hsv_s": 0.7, "hsv_v": 0.4, "mixup": 0.0,
            "mosaic": 1.0, "flipud": 0.0, "weight_decay": 0.0005}
assert all(control_hyp[k] == v for k, v in expected.items())
control_hyp["weight_decay"] = 0.005

def epoch_count(run):
    path = run / "results.csv"
    if not path.is_file():
        return 0
    with path.open(newline="") as stream:
        reader = csv.reader(stream, skipinitialspace=True)
        header = next(reader, [])
        rows = list(reader)
    if any(len(r) != len(header) for r in rows) or [int(r[0]) for r in rows] != list(range(len(rows))):
        raise ValueError(f"Epoch kayıtları eksik veya sırası bozulmuş: {path}")
    return len(rows)

def run_trial(key, recover=False):
    if key not in variants:
        raise ValueError(f"Listeden bir deney seç: {list(variants)}")
    run = sweep / f"{key}_s0"
    hyp = {**control_hyp, **variants[key]}
    hyp_path = work / f"{key}.yaml"
    hyp_path.write_text(yaml.safe_dump(hyp))
    script = "train_close_mosaic.py" if key == "mosaic_son10" else "train.py"
    args = [sys.executable, "-u", script, "--data", str(data_yaml), "--weights", "yolov5s.pt",
            "--img", "640", "--batch", "32", "--epochs", "80", "--seed", "0", "--workers", "8",
            "--cache", "ram", "--hyp", str(hyp_path), "--project", str(sweep), "--name", run.name, "--exist-ok"]
    settings = {"variant": key, "commit": commit, "weight_decay": 0.005, "seed": 0,
                "epochs": 80, "batch_size": 32, "image_size": 640, "weights": "yolov5s.pt",
                "optimizer": "SGD", "workers": 8, "hyp": hyp,
                "mosaic_off_last_epochs": 10 if key == "mosaic_son10" else 0}
    if key == "mosaic_son10":
        settings["launcher_sha256"] = hashlib.sha256((yolo / script).read_bytes()).hexdigest()
    print(f"{key} · {gpu} · {run}", flush=True)
    start = 0
    launch = False
    if not run.exists():
        if recover:
            raise FileNotFoundError("Kurtarılacak deney yok. Yeni deneme için kurtarma kutusunu boş bırak.")
        run.mkdir()
        versions = {}
        for package in ("torch", "torchvision", "ultralytics", "numpy", "albumentations"):
            try:
                versions[package] = metadata.version(package)
            except metadata.PackageNotFoundError:
                versions[package] = None
        manifest = {**settings, "training_gpu": gpu, "cuda": torch.version.cuda,
                    "python": sys.version, "packages": versions, "command": args,
                    "started_utc": datetime.now(timezone.utc).isoformat()}
        (run / "run.json").write_text(json.dumps(manifest, indent=2))
        launch = True
    else:
        manifest = json.loads((run / "run.json").read_text())
        if any(manifest.get(k) != v for k, v in settings.items()):
            raise ValueError(f"Bu klasörde farklı ayarlarda bir koşu var: {run}")
        if (run / "metrics.csv").is_file():
            print("Bu koşunun doğrulaması seçili Drive'da kayıtlı.")
            display_summary(sweep, variants, control)
            return
        start = epoch_count(run)
        if start < 80:
            last = run / "weights/last.pt"
            if not recover:
                raise RuntimeError(f"{start}/80 epoch kayıtlı. Diğer oturumun kapandığından eminsen "
                                   "kurtarma kutusunu işaretleyerek aynı deneyi seç.")
            if not last.is_file() or start == 0:
                raise RuntimeError("Devam edilebilir checkpoint yok. Eski oturum kapalıysa Drive'da bu yarım "
                                   "deney klasörünün adını değiştir ve kurtarma kutusu boşken tekrar başlat.")
            read_epoch = ("import sys,torch; c=torch.load(sys.argv[1],map_location='cpu',weights_only=False); "
                          "print(c['epoch'] if c.get('optimizer') is not None else -1)")
            saved_epoch = int(subprocess.check_output([sys.executable, "-c", read_epoch, str(last)],
                                                      cwd=yolo, text=True).splitlines()[-1])
            if saved_epoch != start - 1:
                raise RuntimeError("last.pt ile results.csv farklı epoch gösteriyor. "
                                   "Bu dosyaları koru ve devam etmeden önce günlüğü kontrol et.")
            args.extend(["--resume", str(last)])
            print(f"{start + 1}. epoch'tan devam ediliyor. Bu oturum değişikliği kayda geçecek.")
            with (run / "resume.jsonl").open("a") as stream:
                stream.write(json.dumps({"completed_epochs": start, "gpu": gpu,
                                         "utc": datetime.now(timezone.utc).isoformat()}) + "\n")
            launch = True
        elif start != 80:
            raise RuntimeError(f"Beklenmeyen epoch sayısı: {start}")
    if launch:
        log = run / "training.log"
        print("Eğitim günlüğü:", log, flush=True)
        env = {**os.environ, "SVRDD_START_EPOCH": str(start)}
        with log.open("ab") as output, subprocess.Popen(args, cwd=yolo, env=env,
                stdout=output, stderr=subprocess.STDOUT) as job:
            previous = start
            try:
                while job.poll() is None:
                    try:
                        completed = epoch_count(run)
                    except ValueError:
                        completed = previous  # YOLO bir CSV satırını o anda yazıyor olabilir.
                    if completed > previous:
                        print(f"{completed}/80 epoch", flush=True)
                        previous = completed
                    time.sleep(5)
            except BaseException:
                job.terminate()
                raise
        if job.returncode:
            with log.open("rb") as stream:
                stream.seek(max(0, log.stat().st_size - 3200))
                print(stream.read().decode(errors="replace"))
            raise RuntimeError(f"Eğitim durdu. Günlük: {log}")
        (run / "training_finished.txt").write_text(datetime.now(timezone.utc).isoformat())
    elif not (run / "training_finished.txt").exists() and not recover:
        raise RuntimeError("80 epoch kayıtlı. Önceki oturum kapandıysa kurtarma kutusuyla yalnızca doğrulamayı çalıştır.")
    if epoch_count(run) != 80:
        raise RuntimeError("80 epoch tamamlanmadan doğrulama yapılamaz.")
    print("best.pt ile 1.000 doğrulama görüntüsü ölçülüyor.", flush=True)
    validate_run(run, yolo=yolo, data_yaml=data_yaml, names=names, control=control, gpu=gpu)
    display_summary(sweep, variants, control)

print("Veri ve deney yapılandırması hazır.")

### Eğitim ve doğrulama

Her çalıştırma bir varyantı eğitir ve en yüksek doğrulama mAP50:95 değerine sahip checkpoint'i değerlendirir. **KURTAR**, kapanmış bir oturumun son tutarlı checkpoint'inden devam eder. 80 epoch tamamlanmışsa yalnızca bekleyen doğrulamayı yürütür.

In [ ]:
#@title 3. Eğitim ve doğrulama
DENEY = "scale_025" #@param ["scale_025", "hsv_s_035", "mixup_010", "mosaic_son10"]
KURTAR = False #@param {type:"boolean"}
run_trial(DENEY, recover=KURTAR)

### Karşılaştırma

Birincil ölçüt, orijinal baseline'da ortalamanın altında kalan dört sınıfın ortalama AP50 değeridir (longitudinal crack, transverse crack, pothole ve transverse patch). Tablodaki farklar, standart augmentation ve weight decay 0.005 ile elde edilen kontrole göre yüzde puan cinsindendir. Genel mAP50:95 ve pothole AP50 ayrıca raporlanır.

Her koşu `metrics.csv` üretir. Dört koşunun sonuçları `augmentation_sweep_results.csv` dosyasında birleşir. Bu karşılaştırma seed 0 için geçerlidir.